# The Scenario

You've joined the ML team at **NorthStar Logistics**. The team has four engineers and ships two models:

1. **ETA model** — predicts arrival time at delivery. Online inference, ~200 RPS, retrained every 2 weeks.
2. **Dispatcher routing model** — picks which courier should accept each pickup. Online inference, ~50 RPS, retrained monthly.

Until now, models have lived in S3 with names like `eta_v2_FINAL.onnx`. There is no registry. The CTO has asked you to fix this.

# Task 1 — Lifecycle diagram

Produce `lifecycle.md` (Mermaid) or `lifecycle.png` showing the end-to-end flow for the ETA model: data → experimentation → training → evaluation → registry → deployment → monitoring → back to data.

Requirements:

- **Name every artifact** on each arrow (dataset hash, run ID, model URI, deployed version, drift signal).
- **Mark the registry stages** (Staging / Production / Archived) as their own boxes.
- **Show the monitoring loopback** — what signal feeds what?
- **Annotate** which transitions are automatic and which require approval.

A good diagram has ~10–15 boxes. If you need 30, you're drawing infrastructure; zoom out to artifacts and contracts.

```mermaid
flowchart TD
    %% Nodes
    Data(Data Prep & Feature Engineering)
    Exp(Experimentation & Prototyping)
    Train(Automated Training Pipeline)
    Eval(Evaluation & Validation)

    subgraph Model Registry
        Staging([Staging])
        Production([Production])
        Archived([Archived])
    end

    Deploy(Online Inference API - 200 RPS)
    Monitor(Monitoring & Observability)

    %% Flow & Artifacts
    Data -- "Dataset Hash & Date Range" --> Exp
    Exp -- "Git SHA & Hyperparameters" --> Train
    Train -- "Run ID + Model URI (S3 Path)" --> Eval
    Eval -- "[Auto] Validation Metrics & Test Scores" --> Staging

    Staging -- "[Manual] Tech Lead Sign-off" --> Production
    Production -- "[Auto] Deprecation Schedule" --> Archived

    Production -- "[Auto] Deployed Version Tag" --> Deploy
    Deploy -- "Prediction Logs & Latency Stats" --> Monitor

    Monitor -- "[Auto] Drift Signal (e.g., MAE > threshold)" --> Train
    Monitor -- "Ground Truth Feedback (Actual Arrivals)" --> Data

    %% Styling
    classDef manual fill:#fdf1ec,stroke:#d9705a,stroke-width:2px;
    classDef auto fill:#eef7ec,stroke:#7eb075,stroke-width:2px;
    classDef registry fill:#eef2ff,stroke:#6366f1,stroke-width:2px;

    class Staging,Production,Archived registry;

### ETA Model Lifecycle Overview

This workflow transitions the ETA model from ad-hoc S3 storage to a structured, observable MLOps pipeline designed to safely handle 200 RPS online inference.

* **Data & Training:** Feature engineering (tracked by dataset hashes) feeds into automated training pipelines (tracked by Git SHAs and Run IDs), ensuring strict reproducibility.
* **Model Registry (Source of Truth):** New models that pass automated evaluation are pushed to **Staging**. The actual `.onnx` binaries remain in S3, but the Registry governs their state.
* **Deployment Gate:** Transitioning a model from Staging to **Production** requires **manual sign-off** to ensure safe deployment for critical dispatch logistics. Older models are automatically scheduled for the **Archived** state.
* **Continuous Monitoring (Feedback Loops):** * **Automated Retraining:** If monitoring detects statistical drift (e.g., MAE exceeds thresholds), an automated signal triggers the training pipeline to generate a fresh model.
    * **Data Enrichment:** Actual courier arrival times are captured as ground-truth feedback and fed directly back into the dataset for future training cycles.

registry:
  name: northstar-models
  stages:
    - name: staging
      promoted_from: automated_training_pipeline
      promotion_rule: automatic
      gates:
        - global_mae_improvement: "Global Mean Absolute Error (MAE) on the frozen evaluation set must be <= current production baseline"
        - strict_latency_check: "Inference latency p99 <= 100ms under simulated 200 RPS load test"
        - extreme_error_bound: "99th percentile of ETA prediction errors must not exceed 15 minutes"

    - name: production
      promoted_from: staging
      promotion_rule: manual
      gates:
        - regional_parity: "Zero degradation > 5% in regional MAE (specifically testing the bottom 3 historically worst-performing delivery zones)"
        - shadow_stability: "Successful 12-hour shadow deployment with 0 Out-Of-Memory (OOM) exceptions and < 0.01% timeout rate"
        - feature_drift_check: "Maximum Population Stability Index (PSI) < 0.1 across all top-10 importance features vs. live traffic"
      approvers:
        - Lead ML Engineer
        - Operations/Dispatch Product Manager

    - name: archived
      promoted_from: production
      promotion_rule: automatic
      trigger: "A new version is promoted to production OR the model version has been inactive in staging for > 30 days"

models:
  - name: eta
    owner: ml-routing-team
    sla_p95_ms: 150
    retention_versions: 10
    
  - name: dispatcher-routing
    owner: ml-dispatch-team
    sla_p95_ms: 80
    retention_versions: 10

lineage:
  required_fields:
    - git_sha
    - dataset_hash
    - training_run_id
    - container_image_uri
    - hyperparameter_signature

### Model Registry Specification Overview

This configuration defines strict, automated governance for the ML lifecycle, ensuring models are not just accurate, but safe for high-stakes logistics operations.

* **Staging Gates (Automated):** Entry to staging is fully automated but guarded by strict performance and system checks. The model must beat the current production baseline (MAE), pass strict latency SLAs (under 100ms at 200 RPS), and avoid extreme outliers (capping 99th percentile errors at 15 minutes) so that edge-case ETAs don't disrupt courier schedules.
* **Production Gates (Manual):** Promotion to production requires human sign-off from both ML and Operations leads. It enforces **slice metrics** (regional parity) to ensure a model doesn't optimize global metrics at the expense of specific delivery zones. It also requires a clean 12-hour shadow deployment to verify system stability under live traffic.
* **Archival:** Deprecation is automated to keep the registry clean, triggering when a new model is promoted or a release candidate stagnates in staging.
* **Lineage Requirements:** To guarantee full reproducibility, every model must be permanently linked to its code (`git_sha`), data (`dataset_hash`), and deployment environment (`container_image_uri`). If a model fails in production, the team can instantly reconstruct the exact conditions that created it.

# Task 2 — Registry specification

Write `model-registry.yaml` defining how the registry behaves. Use this skeleton and fill it in:

```yaml
registry:
  name: northstar-models
  stages:
    - name: staging
      promoted_from: <what triggers entry?>
      promotion_rule: <automatic | manual>
      gates:
        - <metric or check that must pass>
        - <metric or check that must pass>
    - name: production
      promoted_from: staging
      promotion_rule: manual
      gates:
        - <e.g. slice metric thresholds>
        - <e.g. successful canary rollout>
      approvers:
        - <role>
        - <role>
    - name: archived
      promoted_from: <what triggers archival?>

models:
  - name: eta
    owner: <team or role>
    sla_p95_ms: 150
    retention_versions: 10
  - name: dispatcher-routing
    owner: <team or role>
    sla_p95_ms: 80
    retention_versions: 10

lineage:
  required_fields:
    - <e.g. git_sha>
    - <e.g. dataset_hash>
    - <fill in 2-3 more>
```

Be specific. "Tests must pass" is not a gate. "Macro F1 on the frozen evaluation set ≥ 0.82, plus per-city F1 ≥ 0.70 on the bottom 3 cities" is.

# Task 3 — Reproducibility ADR

Write `adr/0001-reproducibility-strategy.md` capturing how the team will guarantee reproducibility across the four layers (environment, data, code, randomness). Use the standard ADR format:

```markdown
# ADR 0001: Reproducibility strategy for NorthStar models

## Context
(Two sentences on the current state — no registry, no dataset versioning, models named with FINAL.)

## Decision
(One paragraph per layer: environment, data, code, randomness — what specifically will be pinned and how.)

## Alternatives rejected
(2–3 bullets — e.g. "no dataset versioning, rely on S3 timestamps" — and why that fails.)

## Consequences
(2–3 bullets — what this commits the team to operationally.)

## Revisit if
(One bullet — what change in scale or compliance forces a rethink.)
```

You **must** name specific tools (or specific lightweight conventions if you choose not to adopt heavy tooling). Generic answers like "we will track experiments somehow" fail the bar.


## ADR 0001: Reproducibility strategy for NorthStar models

## Context
NorthStar currently relies on an unstructured model lifecycle where binary artifacts reside in S3 under naming conventions like `eta_v2_FINAL.onnx` with no linked metadata. Because there is no central registry, dataset versioning, or environment pinning, it is currently impossible to trace a production model back to the exact code, data, and environment that generated it.

## Decision

**Environment**
We will guarantee execution environment reproducibility using **Docker**. Every training run and inference endpoint will execute inside a containerized environment defined by a version-controlled `Dockerfile`. Python dependencies will be strictly pinned using `poetry.lock` to ensure all transitive dependencies remain immutable.

**Data**
We will implement data versioning using **DVC (Data Version Control)** configured with our existing S3 buckets as the remote storage backend. Every training run will record the exact DVC hash corresponding to the data snapshot used, ensuring we can reconstruct the exact feature set, time-window, and ground-truth values ingested during that specific run.

**Code**
All experimentation and training runs will be logged using **MLflow Tracking**. To ensure traceability, our automated training pipeline (running via CI/CD) will strictly enforce that no model can be registered from a dirty Git working tree; every registered model will require a definitive, committed **Git SHA**. 

**Randomness**
We will enforce deterministic training by centralizing random seed configuration. A global configuration utility will set fixed seeds across all stochastic libraries utilized in our stack (e.g., `numpy.random.seed()`, `random.seed()`, and framework-specific seeds like `xgboost.set_config()`). Furthermore, we will explicitly configure our ML frameworks to use deterministic algorithms where applicable to prevent floating-point variances across different CPU/GPU architectures.

## Alternatives rejected
* **Relying on S3 object versioning/timestamps for data:** Rejected because S3 timestamps do not provide atomic, verifiable snapshots of complex, multi-file datasets, nor do they natively link to the codebase state at the time of training.
* **Conda environments or basic `requirements.txt`:** Rejected because cross-platform discrepancies and missing system-level binaries often break environment reproducibility; full Docker containerization isolates us from host-OS variables.
* **Manual tracking via wikis or spreadsheets:** Rejected because manual metadata entry is highly error-prone, doesn't scale to our bi-weekly retraining cycles, and cannot be programmatically validated by deployment gates.

## Consequences
* Engineers can no longer trigger ad-hoc training scripts from their local laptops for production. All release-candidate models must be built via the automated CI/CD pipeline to guarantee Git SHA and container integrity.
* Storage costs will increase moderately as DVC retains historical dataset versions and MLflow stores metadata and artifacts for every experiment.
* Feature engineering code must be updated to output deterministic datasets (e.g., sorting queries before saving to ensure row order doesn't change between runs).

## Revisit if
* The team migrates to a real-time streaming feature store where point-in-time "time-travel" queries (e.g., using Delta Lake or Apache Hudi) become a more native architectural fit for data versioning than DVC file snapshots.